In [1]:
from google.colab import files
uploaded=files.upload()

Saving 1_TestSet_3Class.csv to 1_TestSet_3Class.csv
Saving 1_TrainingSet_3Class.csv to 1_TrainingSet_3Class.csv


In [2]:
import nltk

# Download necessary NLTK resources
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

**Preprocessing**

In [3]:
import pandas as pd


In [4]:
df = pd.read_csv('1_TrainingSet_3Class.csv', header=None)
df.columns = ['label', 'id', 'text']
display(df.head())

,label,id,text
0,2,s801810168862408705,UPDATE: Rt. in York County has reopened near C...
1,0,s899723031702065152,#Fargo #ND #USA - Sr. Clinical Research Associ...
2,0,s840600120727130112,Just two of the biggest fans riding the ride i...
3,1,s900335719461400580,Wreck on Southbound just north of Atlanta High...
4,0,s657684246153314304,Justin bieber's new album is going to be amazi...


In [5]:
df.columns = ['label', 'id', 'text']

In [6]:
import re
from bs4 import BeautifulSoup
import nltk


In [7]:
# Convert the text column to lowercase
df['text'] = df['text'].astype(str).str.lower()



In [8]:
def remove_html_tags(text):
    # Ensure the input is treated as a string before passing to BeautifulSoup
    soup = BeautifulSoup(str(text), "html.parser")
    return soup.get_text()

df['text'] = df['text'].apply(remove_html_tags)

In [9]:
def remove_extra_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

df['text'] = df['text'].apply(remove_extra_whitespace)

In [10]:
def remove_urls(text):
    return re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)

df['text'] = df['text'].apply(remove_urls)

In [11]:
def remove_punctuation_and_digits(text):
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\d+", "", text)
    return text

df['text'] = df['text'].apply(remove_punctuation_and_digits)

In [12]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

def remove_stopwords(text):
    stop_words = set(stopwords.words("english"))
    tokens = word_tokenize(text)
    filtered_tokens = [token for token in tokens if token.lower() not in stop_words]
    return " ".join(filtered_tokens)

df['text'] = df['text'].apply(remove_stopwords)

In [13]:
def tokenize_text(text):
    return nltk.word_tokenize(text)

df['text'] = df['text'].apply(tokenize_text)

In [14]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_text(tokens_list):
    # The 'text' column now contains lists of tokens, not single strings.
    # We iterate directly over the list to lemmatize each token.
    lemmatized = [lemmatizer.lemmatize(word) for word in tokens_list]
    return " ".join(lemmatized)

df['text'] = df['text'].apply(lemmatize_text)

In [15]:
import pandas as pd

test_df = pd.read_csv('1_TestSet_3Class.csv', header=None)
test_df.columns = ['label', 'id', 'text']

display(test_df.head())
test_df['text'] = test_df['text'].astype(str).str.lower()

test_df['text'] = test_df['text'].apply(remove_html_tags)
test_df['text'] = test_df['text'].apply(remove_extra_whitespace)
test_df['text'] = test_df['text'].apply(remove_urls)
test_df['text'] = test_df['text'].apply(remove_punctuation_and_digits)
test_df['text'] = test_df['text'].apply(remove_stopwords)
test_df['text'] = test_df['text'].apply(tokenize_text)
test_df['text'] = test_df['text'].apply(lemmatize_text)

,label,id,text
0,1,s884397880253059074,"As highway construction heats up, please slow ..."
1,1,s902255612217708545,Crash SB Telegraph near Joy Road in Redford To...
2,1,s896765983695794179,Incident on #FLine BothDir/BothDir at Neptune ...
3,1,s900820637140340737,Updated: Crash in Palm Beach on highway north ...
4,0,s840215804671057921,Shared spaces and new amenities are redefining...


In [16]:
# Save back to CSV
df.to_csv("1_TrainingSet_3Class_lower.csv", index=False)

In [17]:
display(df[['text']].head(10))

,text
0,update rt york county reopened near commerce c...
1,fargo nd usa sr clinical research associate sr...
2,two biggest fan riding ride orlando hi jimmy
3,wreck southbound north atlanta highway exit ca...
4,justin biebers new album going amazing
5,next sr manager football development usa footb...
6,exclusive restaurant new york state year wait ...
7,bog flooded water fruit float surface making e...
8,talk nice dont talk
9,cleared disabled vehicle miamidade highway sou...


In [18]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(df)
test_dataset = Dataset.from_pandas(test_df)

In [19]:
from transformers import AutoTokenizer

def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/40942 [00:00<?, ? examples/s]

Map:   0%|          | 0/10236 [00:00<?, ? examples/s]

In [ ]:
Prompt before fine tuning

In [20]:
from transformers import AutoModelForSequenceClassification, pipeline

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer
)

samples = [
{"tweet":"Massive traffic jam after a car accident on highway","label":0},
{"tweet":"Road closed due to construction work downtown","label":1},
{"tweet":"Traffic is moving smoothly this morning","label":2}
]

print("Prompt Testing BEFORE Fine-Tuning\n")

correct = 0

for sample in samples:

    tweet = sample["tweet"]
    true_label = sample["label"]

    prediction = classifier(tweet)[0]

    predicted_label = int(prediction["label"].replace("LABEL_",""))

    verdict = "correct" if predicted_label == true_label else "incorrect"

    if predicted_label == true_label:
        correct += 1

    print("="*60)
    print("Tweet:",tweet)
    print("Predicted:",predicted_label)
    print("Confidence:",prediction["score"])
    print("True:",true_label)
    print("Verdict:",verdict)

accuracy = correct/len(samples)

print("\nAccuracy BEFORE:",accuracy)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Prompt Testing BEFORE Fine-Tuning

Tweet: Massive traffic jam after a car accident on highway
Predicted: 1
Confidence: 0.33860182762145996
True: 0
Verdict: incorrect
Tweet: Road closed due to construction work downtown
Predicted: 1
Confidence: 0.3421606421470642
True: 1
Verdict: correct
Tweet: Traffic is moving smoothly this morning
Predicted: 1
Confidence: 0.3464968502521515
True: 2
Verdict: incorrect

Accuracy BEFORE: 0.3333333333333333


Fine tuning

In [21]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from transformers import EarlyStoppingCallback, TrainerCallback
from peft import LoraConfig, TaskType, get_peft_model
import numpy as np
from sklearn.metrics import accuracy_score

# Model name
model_name = "distilbert-base-uncased"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Base model
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

# LoRA configuration
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_lin", "v_lin"]
)

# Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


# Metric function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}


# Custom checkpoint callback
class CheckpointCallback(TrainerCallback):
    def on_save(self, args, state, control, **kwargs):
        print(f"Checkpoint saved at step {state.global_step}")


# Updated Training arguments
args = TrainingArguments(
    output_dir='./results',
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,

    # --- CHECKPOINT & EVALUATION LOGIC ---
    eval_strategy="steps",           # Evaluate every X steps
    eval_steps=100,                  # How often to run evaluation
    save_strategy="steps",           # MUST match eval_strategy
    save_steps=100,                  # MUST match eval_steps
    load_best_model_at_end=True,     # MUST be True for EarlyStopping
    # -------------------------------------

    save_total_limit=2,              # Keeps the 'best' and the 'latest'
    logging_steps=50,
    metric_for_best_model="accuracy",
    greater_is_better=True,          # Higher accuracy is better
    report_to="none"                 # Prevents unnecessary cloud logging
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 740,355 || all params: 67,696,134 || trainable%: 1.0936


In [23]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2),
        CheckpointCallback()
    ]
)

trainer.train()

Step,Training Loss,Validation Loss,Accuracy
100,0.139012,0.177083,0.951153
200,0.150807,0.182694,0.949980
300,0.175055,0.156209,0.955842
400,0.114115,0.149043,0.958382
500,0.136057,0.140736,0.961508
600,0.142097,0.139690,0.960629
700,0.180445,0.131107,0.962388
800,0.099501,0.141526,0.959066
900,0.141704,0.143369,0.963169
1000,0.099302,0.141826,0.963658


Checkpoint saved at step 100
Checkpoint saved at step 200
Checkpoint saved at step 300
Checkpoint saved at step 400
Checkpoint saved at step 500
Checkpoint saved at step 600
Checkpoint saved at step 700
Checkpoint saved at step 800
Checkpoint saved at step 900
Checkpoint saved at step 1000
Checkpoint saved at step 1100
Checkpoint saved at step 1200
Checkpoint saved at step 1300
Checkpoint saved at step 1400
Checkpoint saved at step 1500
Checkpoint saved at step 1600


TrainOutput(global_step=1600, training_loss=0.12928299903869628, metrics={'train_runtime': 724.9022, 'train_samples_per_second': 169.438, 'train_steps_per_second': 21.181, 'total_flos': 431181221068800.0, 'train_loss': 0.12928299903869628, 'epoch': 0.31262211801484957})

In [24]:
metric_for_best_model="accuracy"
greater_is_better=True

In [25]:
# 1. Get predictions from the trainer
print("\nRunning Evaluation on Test CSV...")
predictions = trainer.predict(tokenized_test)

# 2. Convert raw outputs (logits) to label IDs
# The model outputs 3 numbers; np.argmax picks the highest one
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

# 3. Calculate Final Accuracy on the Test Set
test_accuracy = accuracy_score(labels, preds)
print(f"Final Test Accuracy: {test_accuracy:.2%}")


Running Evaluation on Test CSV...


Final Test Accuracy: 96.82%


prompt after fune tuning

In [26]:
from transformers import pipeline

print("\nPrompt Testing AFTER Fine-Tuning (from Test CSV)\n")

# Use the fine-tuned model
classifier = pipeline(
    "text-classification",
    model=trainer.model,
    tokenizer=tokenizer
)

correct = 0
num_samples = 10   # number of test examples to show

for i in range(num_samples):

    tweet = test_df.iloc[i]["text"]
    true_label = int(test_df.iloc[i]["label"])

    prediction = classifier(tweet)[0]
    predicted_label = int(prediction["label"].replace("LABEL_", ""))

    verdict = "correct" if predicted_label == true_label else "incorrect"

    if predicted_label == true_label:
        correct += 1

    print("="*60)
    print("Tweet:", tweet)
    print("Predicted:", predicted_label)
    print("Confidence:", prediction["score"])
    print("True Label:", true_label)
    print("Verdict:", verdict)

accuracy = correct / num_samples
print("\nPrompt Accuracy AFTER Fine-Tuning:", accuracy)


Prompt Testing AFTER Fine-Tuning (from Test CSV)

Tweet: highway construction heat please slow aware workzones work zone safety hand
Predicted: 0
Confidence: 0.7301371693611145
True Label: 1
Verdict: incorrect
Tweet: crash sb telegraph near joy road redford township affecting left lane traffic slow plymouth road wwjtraffic
Predicted: 1
Confidence: 0.99989914894104
True Label: 1
Verdict: correct
Tweet: incident fline bothdirbothdir neptune avenue station
Predicted: 1
Confidence: 0.9999278783798218
True Label: 1
Verdict: correct
Tweet: updated crash palm beach highway north exit yamato rd left lane blocked last updated pm
Predicted: 1
Confidence: 0.9999513626098633
True Label: 1
Verdict: correct
Tweet: shared space new amenity redefining way office facilitate community creativity
Predicted: 0
Confidence: 0.9999172687530518
True Label: 0
Verdict: correct
Tweet: disabled vehicle right lane blocked vanalystnegraysoncoline nb cnty line rd stop go traffic back fm dfwtraffic
Predicted: 1
Conf

Comparastion before and After fine tuning

In [28]:
from transformers import AutoModelForSequenceClassification, pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

print("\nPerformance Comparison: BEFORE vs AFTER Fine-Tuning\n")

# ==============================
# LOAD BASE MODEL AGAIN
# ==============================

base_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3
)

base_classifier = pipeline(
    "text-classification",
    model=base_model,
    tokenizer=tokenizer
)

# ==============================
# BEFORE FINE-TUNING
# ==============================

before_preds = []
true_labels = []

for example in test_dataset:

    text = example["text"]
    label = example["label"]

    prediction = base_classifier(text)[0]

    pred_label = int(prediction["label"].replace("LABEL_",""))

    before_preds.append(pred_label)
    true_labels.append(label)

before_accuracy = accuracy_score(true_labels, before_preds)

before_precision, before_recall, before_f1, _ = precision_recall_fscore_support(
    true_labels,
    before_preds,
    average="macro"
)

# ==============================
# AFTER FINE-TUNING
# ==============================

predictions = trainer.predict(tokenized_test)

after_preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

after_accuracy = accuracy_score(labels, after_preds)

after_precision, after_recall, after_f1, _ = precision_recall_fscore_support(
    labels,
    after_preds,
    average="macro"
)

# ==============================
# PRINT RESULTS
# ==============================

print("Metric        BEFORE Fine-Tuning    AFTER Fine-Tuning")
print("-----------------------------------------------------")
print(f"Accuracy      {before_accuracy:.4f}                {after_accuracy:.4f}")
print(f"Precision     {before_precision:.4f}                {after_precision:.4f}")
print(f"Recall        {before_recall:.4f}                {after_recall:.4f}")
print(f"F1 Score      {before_f1:.4f}                {after_f1:.4f}")


Performance Comparison: BEFORE vs AFTER Fine-Tuning



Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Metric        BEFORE Fine-Tuning    AFTER Fine-Tuning
-----------------------------------------------------
Accuracy      0.3339                0.9682
Precision     0.1495                0.9640
Recall        0.2348                0.9581
F1 Score      0.1825                0.9609


Final evaluition by testing

In [30]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from transformers import pipeline

# Use the fine-tuned model
classifier = pipeline(
    "text-classification",
    model=trainer.model,
    tokenizer=tokenizer
)

# Prepare test labels and predictions
true_labels = test_df['label'].tolist()
pred_labels = []

for text in test_df['text']:
    pred = classifier(text)[0]
    pred_label = int(pred["label"].replace("LABEL_", ""))
    pred_labels.append(pred_label)

# Compute metrics
accuracy = accuracy_score(true_labels, pred_labels)
precision, recall, f1, _ = precision_recall_fscore_support(true_labels, pred_labels, average="macro")
cm = confusion_matrix(true_labels, pred_labels)

# Print metrics
print("Evaluation on Test Set AFTER Fine-Tuning")
print("=========================================")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")



Evaluation on Test Set AFTER Fine-Tuning
Accuracy : 0.9682
Precision: 0.9640
Recall   : 0.9581
F1 Score : 0.9609


Test is their a data leakage

In [31]:
# Check for exact matches between Train and Test
train_texts = set(df['text'])
test_texts = set(test_df['text'])

overlap = train_texts.intersection(test_texts)

print(f"Number of overlapping tweets: {len(overlap)}")
if len(overlap) > 0:
    print("Warning: Data Leakage detected. Some tweets are in both sets.")
else:
    print("Clean: No identical tweets found between Train and Test.")

Number of overlapping tweets: 203


In [32]:
# 1. Create a clean test set by removing training overlaps
train_texts = set(df['text'])
test_df_honest = test_df[~test_df['text'].isin(train_texts)].copy()

# 2. Convert to Dataset and Tokenize
from datasets import Dataset
honest_test_dataset = Dataset.from_pandas(test_df_honest)
tokenized_honest_test = honest_test_dataset.map(tokenize_function, batched=True)

# 3. Predict using the ALREADY TRAINED model
print("Calculating Honest Performance (Zero Leakage)...")
honest_results = trainer.predict(tokenized_honest_test)
honest_preds = np.argmax(honest_results.predictions, axis=1)
honest_labels = honest_results.label_ids

# 4. Final Final Metrics
from sklearn.metrics import classification_report, accuracy_score
print(f"Honest Accuracy: {accuracy_score(honest_labels, honest_preds):.4f}")
print("\nFinal Classification Report:")
print(classification_report(honest_labels, honest_preds, target_names=["Incident", "Condition", "Non-Traffic"]))

Map:   0%|          | 0/10033 [00:00<?, ? examples/s]

Calculating Honest Performance (Zero Leakage)...


Honest Accuracy: 0.9679

Final Classification Report:
              precision    recall  f1-score   support

    Incident       0.98      0.98      0.98      5111
   Condition       0.95      0.97      0.96      3385
 Non-Traffic       0.95      0.92      0.94      1537

    accuracy                           0.97     10033
   macro avg       0.96      0.96      0.96     10033
weighted avg       0.97      0.97      0.97     10033

